In [2]:

# imports required libraries.
import requests
import json
import time
import pandas as pd
from datetime import datetime




In [3]:
# Set your TomTom API key.
tomtom_key = "Pmi9QjFCM7d2MFaDmqQ4zjOBGGV0muSF"  # Replace with your key


In [4]:
# Define a coordinate grid for the search area.
# (Here we use a few latitude and longitude points for the Copenhagen area.)
lats = [55.663, 55.664, 55.665, 55.666]
lons = [12.551, 12.552, 12.553, 12.554, 12.555, 12.556, 12.557, 12.558, 12.559, 12.56]


In [5]:
# Query the EV charging stations using the "poiSearch/charging.json" endpoint.
stations = []
for lat in lats:
    for lon in lons:
        print(f"Querying for coordinates: lat={lat}, lon={lon}")
        url = (
            f"https://api.tomtom.com/search/2/poiSearch/charging.json?key={tomtom_key}"
            f"&lat={lat}&lon={lon}&radius=10000&limit=100"
        )
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            num = data.get("summary", {}).get("numResults", 0)
            print(f"  Found {num} results")
            # Append each result from this search to the stations list.
            stations.extend(data.get("results", []))
        else:
            print("  Error:", response.status_code)
        time.sleep(0.2)  # Respect API rate limits

print("Total number of EV charging stations found:", len(stations))

Querying for coordinates: lat=55.663, lon=12.551
  Found 1 results
Querying for coordinates: lat=55.663, lon=12.552
  Found 1 results
Querying for coordinates: lat=55.663, lon=12.553
  Found 1 results
Querying for coordinates: lat=55.663, lon=12.554
  Found 1 results
Querying for coordinates: lat=55.663, lon=12.555
  Found 1 results
Querying for coordinates: lat=55.663, lon=12.556
  Found 1 results
Querying for coordinates: lat=55.663, lon=12.557
  Found 1 results
Querying for coordinates: lat=55.663, lon=12.558
  Found 1 results
Querying for coordinates: lat=55.663, lon=12.559
  Found 1 results
Querying for coordinates: lat=55.663, lon=12.56
  Found 1 results
Querying for coordinates: lat=55.664, lon=12.551
  Found 1 results
Querying for coordinates: lat=55.664, lon=12.552
  Found 1 results
Querying for coordinates: lat=55.664, lon=12.553
  Found 1 results
Querying for coordinates: lat=55.664, lon=12.554
  Found 1 results
Querying for coordinates: lat=55.664, lon=12.555
  Found 1 resu

In [7]:
# Create a DataFrame of selected station fields.
df = pd.DataFrame([{
    "id": s.get("id"),
    "name": s.get("poi", {}).get("name"),
    "phone": s.get("poi", {}).get("phone"),
    "address": s.get("address", {}).get("freeformAddress"),
    "lat": s.get("position", {}).get("lat"),
    "lon": s.get("position", {}).get("lon"),
    "score": s.get("score"),
    "distance": s.get("dist")
} for s in stations])

print("Sample EV Charging Station Data:")
print(df.head())

# Save the station data to a CSV file.
df.to_csv("ev_charging_stations.csv", index=False)

Sample EV Charging Station Data:
                       id             name            phone  \
0  ft6l2oLeuOMVzf0oU-JL8w  Norlys Charging  +45 70 11 50 00   
1  ft6l2oLeuOMVzf0oU-JL8w  Norlys Charging  +45 70 11 50 00   
2  ft6l2oLeuOMVzf0oU-JL8w  Norlys Charging  +45 70 11 50 00   
3  ft6l2oLeuOMVzf0oU-JL8w  Norlys Charging  +45 70 11 50 00   
4  ft6l2oLeuOMVzf0oU-JL8w  Norlys Charging  +45 70 11 50 00   

                                    address        lat        lon     score  \
0  Nattergalevej 6, 2400 København Nordvest  55.697675  12.531828  0.889025   
1  Nattergalevej 6, 2400 København Nordvest  55.697675  12.531828  0.889025   
2  Nattergalevej 6, 2400 København Nordvest  55.697675  12.531828  0.889025   
3  Nattergalevej 6, 2400 København Nordvest  55.697675  12.531828  0.888553   
4  Nattergalevej 6, 2400 København Nordvest  55.697675  12.531828  0.888553   

      distance  
0  4038.682669  
1  4057.781686  
2  4077.755204  
3  4098.590438  
4  4120.274316  


In [8]:

# (Optional) For each station, query its availability.
# The chargingAvailability endpoint expects a parameter "chargingAvailability" equal to the station id.
# Note: In our testing the response may contain an empty "connectors" list.
availability_data = []
for s in stations:
    station_id = s.get("id")
    if station_id:
        url_avail = (
            f"https://api.tomtom.com/search/2/chargingAvailability.json?key={tomtom_key}"
            f"&chargingAvailability={station_id}"
        )
        print("Querying availability for station", station_id)
        resp_avail = requests.get(url_avail)
        if resp_avail.status_code == 200:
            avail = resp_avail.json()
            availability_data.append({
                "station_id": station_id,
                "availability_response": avail,
                "timestamp": datetime.now()
            })
            print("  Response:", avail)
        else:
            print("  Availability query error:", resp_avail.status_code)
        time.sleep(0.2)

# Optionally, you can create a DataFrame of the availability responses.
if availability_data:
    df_avail = pd.DataFrame(availability_data)
    df_avail.to_csv("ev_charging_availability.csv", index=False)
    print("Availability data saved.")
else:
    print("No availability data was returned.")



Querying availability for station ft6l2oLeuOMVzf0oU-JL8w
  Response: {'connectors': [], 'chargingAvailability': 'ft6l2oLeuOMVzf0oU-JL8w'}
Querying availability for station ft6l2oLeuOMVzf0oU-JL8w
  Response: {'connectors': [], 'chargingAvailability': 'ft6l2oLeuOMVzf0oU-JL8w'}
Querying availability for station ft6l2oLeuOMVzf0oU-JL8w
  Response: {'connectors': [], 'chargingAvailability': 'ft6l2oLeuOMVzf0oU-JL8w'}
Querying availability for station ft6l2oLeuOMVzf0oU-JL8w
  Response: {'connectors': [], 'chargingAvailability': 'ft6l2oLeuOMVzf0oU-JL8w'}
Querying availability for station ft6l2oLeuOMVzf0oU-JL8w
  Response: {'connectors': [], 'chargingAvailability': 'ft6l2oLeuOMVzf0oU-JL8w'}
Querying availability for station ft6l2oLeuOMVzf0oU-JL8w
  Response: {'connectors': [], 'chargingAvailability': 'ft6l2oLeuOMVzf0oU-JL8w'}
Querying availability for station ft6l2oLeuOMVzf0oU-JL8w
  Response: {'connectors': [], 'chargingAvailability': 'ft6l2oLeuOMVzf0oU-JL8w'}
Querying availability for station 